# 독립 아이템 N/V 축 + 축 블록 dropout M2 체크포인트 진단

이미 학습된 seed 42 M1·M2 체크포인트만 불러옵니다. **재학습과 checkpoint 선택은 하지 않습니다.**

출력: ID-only / ID+N / ID+V / full의 전체·저·중·고CLV 성과, 각 축이 Top-10·20·50 정답을 넣고 뺀 개수, 실제 후보점수 영향력.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '838e63aade360c1634a317208fcd548ebc3dc05f'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('진단 코드 SHA 확인:', REVIEWED_SHA)

In [ ]:
import json
import torch
from lightgcn_clv_gatefree_lowdim_independent_dropout_diagnostic import (
    configure_independent_dropout_diagnostic,
    preflight_summary,
    run_independent_dropout_diagnostic,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_independent_dropout_diagnostic(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_gatefree_lowdim_independent_dropout_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
    eval_batch_size=32,
)
summary = preflight_summary(cfg)
assert summary['training'] is False
assert summary['checkpoint_selection'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# 재학습 없음: 저장된 M1·M2 체크포인트만 평가
report = run_independent_dropout_diagnostic(cfg)

In [ ]:
from IPython.display import display

print('1) 어느 축이 전체·CLV 구간 성과를 만들었는지')
display(report['axis_effect'])

print('2) 각 축이 Top-K 정답을 넣고 뺀 개수')
display(report['rank_flow'])

print('3) ID/N/V/full 절대지표')
display(report['view_metrics'])

print('4) ID 대비 N/V 실제 점수 영향력')
display(report['score_strength'])

print('저장 파일')
print(json.dumps(report['paths'], ensure_ascii=False, indent=2))